This is basically a simplified re-implementation of section 3 of: [Attention is All You Need](https://arxiv.org/pdf/1706.03762)

Covering only for now:
Emb + Pos_Emb -> Attention

In [49]:
from pathlib import Path
import random
import math

# __file__ isn't defined in notebooks; Jupyter's cwd is the notebook's own folder.
DATA_DIR = Path.cwd().resolve().parent.parent / "bengio2003" / "data"


def read_vocab(path=DATA_DIR / "vocab.txt"):
    """Read vocab.txt into a list where index == token id."""
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]


def read_ids(path=DATA_DIR / "train.ids"):
    """Read a whitespace-separated .ids file into a list of ints."""
    with open(path, encoding="utf-8") as f:
        return [int(tok) for tok in f.read().split()]

vocab = read_vocab()
ids = read_ids()

In [50]:
context = 20
model_size = 50
q_k_v_size = 10
n_heads = 5

In [ ]:
# split the ids into non-overlapping windows 
context_split = [
    ids[i : i + context + 1] for i in range(0, len(ids) - context, context + 1)
]
# this way you have ids[0:21], ids[21:42] - length is always 21 as the last token is the final target, so context used is 20

In [ ]:
# initialize embedding matrix with random values
def create_layer(rows, cols):
    """Create a random layer weight matrix of shape (rows, cols)."""
    return [[random.uniform(-0.1, 0.1) for _ in range(cols)] for _ in range(rows)]

embedding_matrix = create_layer(len(vocab), model_size)

# initialize positional embedding matrix with random values (one vector for each position in context)
# people don't do this anymore, but the idea was to learn a representation for each position
positional_embedding_matrix = create_layer(context, model_size)

In [ ]:
# one independent set of q/k/v projections per head
# we assume all qkv have same size
q_matrices = [create_layer(q_k_v_size, model_size) for _ in range(n_heads)]
k_matrices = [create_layer(q_k_v_size, model_size) for _ in range(n_heads)]
v_matrices = [create_layer(q_k_v_size, model_size) for _ in range(n_heads)]

# to project the concatenated heads 
out_projection_matrix = create_layer(model_size, model_size)

In [ ]:
def compute_attention(last_token_embedding, context_embeddings, q_matrix, k_matrix, v_matrix, q_k_v_size):

    # we compute q only for the new token, and k and v for the entire context
    q = []
    for feature_weights in q_matrix:
        projection = sum(w * e for w, e in zip(feature_weights, last_token_embedding))
        q.append(projection)

    keys = []
    # this is equivalent to the triangular matrix, you only see
    # up to the current token
    for context_embedding in context_embeddings:
        k = []
        for feature_weights in k_matrix:
            projection = sum(w * e for w, e in zip(feature_weights, context_embedding))
            k.append(projection)
        keys.append(k)

    values = []
    for context_embedding in context_embeddings:
        v = []
        for feature_weights in v_matrix:
            projection = sum(w * e for w, e in zip(feature_weights, context_embedding))
            v.append(projection)
        values.append(v)

    # dot product of q and k, scaled by 1/sqrt(d_k).
    # in a batched matrix implementation this is Q @ K.T / sqrt(d_k)
    # look at section 3.2.1, first thet describe it like this
    # then they say in practice you do a single mat mul
    scale = math.sqrt(q_k_v_size)
    dot_products = []
    for k in keys:
        dot_product = sum(q_i * k_i for q_i, k_i in zip(q, k)) / scale
        dot_products.append(dot_product)

    # softmax of dot products to get probabilities.
    exps = [math.exp(s) for s in dot_products]
    z = sum(exps)
    probs = [e / z for e in exps]

    # weighted sum of values using the probabilities
    final_representation = [0.0] * len(values[0]) 
    for prob, v in zip(probs, values):
        for i in range(len(values[0])):
            final_representation[i] += prob * v[i]
    return final_representation

In [ ]:
context_zero = context_split[0]
# at training time you dont do this sequentially, this is the entire point of the attention mechanism
# you compute the attention for all the tokens in the context at once,
# but here I am doing it sequentially to illustrate the process.
for t in range(1, len(context_zero)):
    target_id = context_zero[t]
    context_ids = context_zero[:t]
    context_embeddings = [embedding_matrix[idx] for idx in context_ids]

    # add positional embeddings to the context embeddings
    # now people don't use anymore, they use rope inside attention
    for i in range(len(context_embeddings)):
        context_embeddings[i] = [e + p for e, p in zip(context_embeddings[i], positional_embedding_matrix[i])]

    last_token_embedding = context_embeddings[-1]
    # at inference time you would cache the k and v for the context, and only compute q for the new token

    # each head has its own q/k/v projections, so it can attend to different things
    head_outputs = []
    for head in range(n_heads):
        head_output = compute_attention(
            last_token_embedding, context_embeddings,
            q_matrices[head], k_matrices[head], v_matrices[head],
            q_k_v_size,
        )
        head_outputs.append(head_output)

    # concatenate the outputs of all heads
    concatenated_output = []
    for head_output in head_outputs:
        concatenated_output.extend(head_output)

    # project the concatenated output using out_projection_matrix that information gets mixed across heads
    final_output = []
    for out_row in out_projection_matrix:
        projected = sum(w * c for w, c in zip(out_row, concatenated_output))
        final_output.append(projected)

    # add the resiudal
    final_output = [f + l for f, l in zip(final_output, last_token_embedding)]

    # layer norm (now this happens before attention, but in 2017 paper it was done here)
    # ...